In [0]:
Retrieval Augmented Generation 

Why?
1. LLM have cut off date?
2. LLM do not have org data (Context ) (no knowledge)
3. But there is no reference 
4. 



phase 1. Data Prep


PDF, Text, docs, ppts, etc  (Unstructured data)

1. Extract the data (Text) ( text loader, langchain , )
2. chunking  (python, langchain)
3. Embedding (Vectors)  text to number vector (embedding models) (native embedding modle )
4. Vector databases. (Pinecone, Milvus, chromadb postgre ) (Vector store)





Phase: RAG


user---- ------(LLM)


In [0]:
%pip install -U langchain langchain-community databricks-langchain
 
dbutils.library.restartPython()

# Data Preparation

In [0]:
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter

print("=" * 70)
print("DOCUMENT LOADER EXAMPLE 1: Load & Split Text")
print("=" * 70)
print()

# Sample text (simulating a loaded document)
sample_text = """Artificial Intelligence (AI) is transforming the world. 
Machine Learning is a subset of AI that focuses on learning from data.
Deep Learning is a specialized form of Machine Learning using neural networks.
Natural Language Processing helps computers understand human language.
Computer Vision enables machines to interpret visual information."""

print("Original Text:")
print(sample_text)
print(f"\nTotal Characters: {len(sample_text)}")

print("\n" + "=" * 70)

# Create a Document object
document = Document(page_content=sample_text, metadata={"source": "AI_overview.txt"})

print("\n📄 Document Created:")
print(f"  Content Preview: {document.page_content[:50]}...")
print(f"  Metadata: {document.metadata}")

print("\n" + "=" * 70)
print("✅ Document loaded successfully!")
print("=" * 70)

In [0]:
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

print("=" * 70)
print("TEXT SPLITTER EXAMPLE 2: Character vs Recursive Splitting")
print("=" * 70)
print()

# Long sample text
long_text = """Python is a high-level programming language. It was created by Guido van Rossum.

Python is known for its simplicity and readability. The language emphasizes code readability.

Python supports multiple programming paradigms. These include object-oriented and functional programming.

Python has a large standard library. This makes it suitable for many applications.

Python is widely used in data science. Machine learning libraries like TensorFlow use Python."""

print(f"Original Text Length: {len(long_text)} characters")
print(f"Original Text:\n{long_text}")

print("\n" + "=" * 70)
print("\n🔪 METHOD 1: CharacterTextSplitter")
print("Splits on a specific separator (e.g., newline)\n")

# Character splitter - splits on separator
char_splitter = CharacterTextSplitter(
    separator="\n\n",  # Split on double newline
    chunk_size=100,     # Max chunk size
    chunk_overlap=20    # Overlap between chunks
)

char_chunks = char_splitter.split_text(long_text)

print(f"Number of Chunks: {len(char_chunks)}")
for i, chunk in enumerate(char_chunks, 1):
    print(f"\nChunk {i} ({len(chunk)} chars):")
    print(f"  {chunk[:80]}...")

print("\n" + "=" * 70)
print("\n🔪 METHOD 2: RecursiveCharacterTextSplitter (RECOMMENDED)")
print("Tries multiple separators in order: \\n\\n, \\n, space, character\n")

# Recursive splitter - tries multiple separators
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]  # Order matters!
)

recursive_chunks = recursive_splitter.split_text(long_text)

print(f"Number of Chunks: {len(recursive_chunks)}")
for i, chunk in enumerate(recursive_chunks, 1):
    print(f"\nChunk {i} ({len(chunk)} chars):")
    print(f"  {chunk[:100]}...")

print("\n" + "=" * 70)
print("✅ Text splitting complete!")
print("\n💡 TIP: RecursiveCharacterTextSplitter is better for most cases")
print("It tries to keep paragraphs, sentences, and words together.")
print("=" * 70)

In [0]:
from databricks_langchain import DatabricksEmbeddings

embeddings=DatabricksEmbeddings(endpoint="databricks-gte-large-en")

text=input("Enter some text: ")
response=embeddings.embed_query(text)
print(response)

In [0]:
import numpy as np

In [0]:
from langchain_openai import OpenAIEmbeddings

embeddings1=DatabricksEmbeddings(endpoint="databricks-bge-large-en")
embeddings2=DatabricksEmbeddings(endpoint="databricks-bge-large-en")


text1=input("Enter some text: ")
text2=input("Enter some text: ")

response1=embeddings1.embed_query(text1)
response2=embeddings2.embed_query(text2)

similarity=np.dot(response1, response2)
print(similarity*100,'%')

In [0]:
from langchain_openai import OpenAIEmbeddings

embeddings1=DatabricksEmbeddings(endpoint="databricks-bge-large-en")
embeddings2=DatabricksEmbeddings(endpoint="databricks-gte-large-en")


text1=input("Enter some text: ")
text2=input("Enter some text: ")

response1=embeddings1.embed_query(text1)
response2=embeddings2.embed_query(text2)

similarity=np.dot(response1, response2)
print(similarity*100,'%')

In [0]:
from langchain_openai import OpenAIEmbeddings

llm=DatabricksEmbeddings(endpoint="databricks-bge-large-en")
embeddings=DatabricksEmbeddings(endpoint="databricks-gte-large-en")


text1=input("Enter some text: ")
text2=input("Enter some text: ")

response1=llm.embed_query(text1)
response2=embeddings.embed_query(text2)

similarity=np.dot(response1, response2)
print(similarity*100,'%')

In [0]:
from databricks_langchain import ChatDatabricks
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import os

print("=" * 70)
print("REAL-WORLD USE CASE: HR Resume Analyzer")
print("=" * 70)
print()

# ============================================================================
# SETUP: Volume Path for Resume Storage
# ============================================================================
RESUME_VOLUME_PATH = "/Volumes/main/default/resumes"
# Note: You need to:
# 1. Create Unity Catalog volume: CREATE VOLUME main.default.resumes
# 2. Upload resume files (PDF/TXT) to this volume
# ============================================================================

print("📁 Configuration:")
print(f"  Resume Storage: {RESUME_VOLUME_PATH}")
print(f"  Supported Formats: PDF, TXT")
print()

# For demonstration, we'll simulate resume text
# In production, you would use PyPDFLoader or TextLoader from the volume
sample_resume = """JOHN DOE
Email: john.doe@email.com | Phone: (555) 123-4567

PROFESSIONAL SUMMARY
Experienced Data Engineer with 5+ years of expertise in building scalable data pipelines.
Proficient in Python, Spark, SQL, and cloud platforms (AWS, Azure).

WORK EXPERIENCE

Senior Data Engineer | TechCorp Inc. | 2021 - Present
- Designed and implemented ETL pipelines processing 10TB+ daily data
- Built real-time streaming solutions using Apache Kafka and Spark Streaming
- Optimized SQL queries reducing processing time by 40%
- Led migration from on-premise to AWS cloud infrastructure

Data Engineer | DataSolutions LLC | 2019 - 2021
- Developed Python-based data transformation workflows
- Created automated testing frameworks for data quality
- Collaborated with data scientists on ML feature engineering

SKILLS
Programming: Python, SQL, Scala, Java
Big Data: Apache Spark, Hadoop, Kafka
Cloud: AWS (S3, EMR, Redshift), Azure (Databricks, Synapse)
Databases: PostgreSQL, MySQL, MongoDB
Tools: Git, Docker, Airflow, DBT

EDUCATION
Bachelor of Science in Computer Science
University of Technology | 2015 - 2019

CERTIFICATIONS
- AWS Certified Data Analytics - Specialty
- Databricks Certified Data Engineer Professional"""

print("📝 Processing Resume...\n")
print("=" * 70)

# Step 1: Create document
resume_doc = Document(
    page_content=sample_resume,
    metadata={"source": "john_doe_resume.pdf", "candidate_name": "John Doe"}
)

print(f"\n📄 Document Loaded:")
print(f"  Source: {resume_doc.metadata['source']}")
print(f"  Length: {len(resume_doc.page_content)} characters")

# Step 2: Split text for processing
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents([resume_doc])
print(f"  Chunks Created: {len(chunks)}")

print("\n" + "=" * 70)
print("\n🤖 Analyzing Resume with AI...\n")

# Step 3: Extract structured information using LLM
llm = ChatDatabricks(model="databricks-gemma-3-12b")
json_parser = JsonOutputParser()

extraction_prompt = PromptTemplate(
    input_variables=["resume_text"],
    template="""Extract key information from this resume in JSON format.

{format_instructions}

Return JSON with these fields:
- name: candidate's full name
- email: email address
- total_experience_years: total years of experience (number)
- current_position: current job title
- key_skills: array of top 5 skills
- education: highest degree
- certifications: array of certifications

Resume:
{resume_text}""",
    partial_variables={"format_instructions": json_parser.get_format_instructions()}
)

extraction_chain = extraction_prompt | llm | json_parser

# Extract information
candidate_info = extraction_chain.invoke({"resume_text": sample_resume})

print("📊 Extracted Information:\n")
print(f"  Name: {candidate_info.get('name')}")
print(f"  Email: {candidate_info.get('email')}")
print(f"  Experience: {candidate_info.get('total_experience_years')} years")
print(f"  Current Role: {candidate_info.get('current_position')}")
print(f"  Education: {candidate_info.get('education')}")
print(f"\n  Top Skills:")
for skill in candidate_info.get('key_skills', []):
    print(f"    - {skill}")
print(f"\n  Certifications:")
for cert in candidate_info.get('certifications', []):
    print(f"    - {cert}")

print("\n" + "=" * 70)

# Step 4: Generate job match score
match_prompt = PromptTemplate(
    input_variables=["resume_text", "job_requirements"],
    template="""Rate how well this candidate matches the job requirements.

Job Requirements:
{job_requirements}

Candidate Resume:
{resume_text}

Provide a match score (0-100) and brief reasoning (2 sentences).
Format: Score: XX\nReasoning: ..."""
)

match_chain = match_prompt | llm

job_requirements = """Looking for: Senior Data Engineer
Required: 5+ years experience, Python, Spark, SQL, AWS
Preferred: Databricks certification, Kafka experience"""

match_result = match_chain.invoke({
    "resume_text": sample_resume,
    "job_requirements": job_requirements
})

print("\n🎯 Job Match Analysis:\n")
print(match_result.content)

print("\n" + "=" * 70)
print("✅ Resume analysis complete!")
print("=" * 70)

print("\n" + "*" * 70)
print("HOW TO USE IN PRODUCTION:")
print(f"1. Upload resumes to: {RESUME_VOLUME_PATH}")
print("2. Use PyPDFLoader for PDF files:")
print("   from langchain_community.document_loaders import PyPDFLoader")
print(f"   loader = PyPDFLoader('{RESUME_VOLUME_PATH}/resume.pdf')")
print("   documents = loader.load()")
print("3. Process multiple resumes in batch")
print("4. Store results in Delta table for analytics")
print("*" * 70)